# Neden Embedding? Kelime değil, Anlam ara

- Müşteri "ürünüm bozuk geldi" Belgede "arızalı ürün iadesi"

- Ctrl+F / kelime arama bunu bulamaz -- ortak kelime yok, ama anlam aynı,

- Embedding her metin bir "anlam koordinatı'na" çevirir -> yakın anlam = yakın koordinat

- Artık "kelime eşleşen'i" değil, "anlamca en yakın'ı" getiririz. RAG'in retrieval'ı buna dayanır.

- Analoji tüm metinleri bir haritaya koy "iade" <--> "geri gönderim yan yana "iade" <--> "kargo ücretli" daha uzak 

# Embedding Nedir?

- Metin -> sabit uzunlukta yoğun vektör

- Word2Vec: tek kelime (300 boyut)

- Sentence/Doc embeddings: tüm metin (384-1536)

- Semantik yakınlık -> vektör mesafesi

# 1. Chunk'ları hazırla (nl'in özeti)

In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_tet_splitters import RecursiveCharacterTextSplitter
docs = DirectoryLoader("şirket-bilgi-bankasi/", glob = "**/*.md", loader_cls = TextLoader).load()
chunks = RecursiveCharacterTextSplitter(chunks_size = 400, chunk_overlap = 60).split_documents(docs)
KATEGORİ = {
    "kargo_teslimat": "Kargo & Teslimat", "iade & Değişim": "İade & Değişim",
    "siparis-odeme": "Siparis & Ödeme", "uyelik-sadakat":"Üyelik & Sadakat",
    "sss": "Sıkça Sorulan Sorular"
}

# 2. Embedding Modeli (OpenAI)

- Metni sabit uzunlukta vektöre çevirir. Anlanca yakın metinler -> vektör uzayında yakın.

In [ ]:
load_dotenv()
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small") # 1536 boyutlu vektör

# Ücretsiz alternatif (key istemez) :

# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
vec = embeddings.embed_query("iade süresi kaç gün?")
print("Vektör boyutu:", len(vec), "| ilk 5:", [round(x,3) for x in vec[:5]])

# İki metin ne kadar benzer? (cosine)

- Benzerlik = vektorler arası açı (normalize dot product). 1'e yakın = aynı anlam. Pratikte elle yazmayın -- cosine_similarity (sklearrn) ya da vektör DB hesaplar.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

v1 = embeddings.embed_query("ürünü geri gönderebilir miyim?")
v2 = embeddings.embed_query("iade işlemi nasıl yapılır")
v3 = embeddings.embed_query("bügün hava nasıl?")

print("yakın", round(float(cosine_similarity([v1], [v2])[0][0]),2))
print("uzak", round(float(cosine_similarity([v1], [v3])[0][0]),2))

# Özet 

- Embedding = metin -> vektör (1536)

- Cosine similarity = semantik yakınlık

- OpenAI text-embedding-3-small default (.env); HF all-MiniLM ücretsiz alternatif

- Open-source alternatif: sentence-transformers

# Vektör DB Neden

- Milyonlarca embedding'i saklayıp aramak için

- Hızlı similarity search (HNSW, IVF index)

- Metadata filtering

- Persistent storage

- Farklı bilgi tabanlarını ayrı tut (farklı collection)

# Chroma -> Pinecone geçişi

- Chroma = öğren / local/ bedava ---  Pinecone = production / managed cloud

- Önemli: retrieval interface'i AYNI -- store.as_retriever() değişmez, sadace storu değiştirir.

# 3. Chroma vektör DB'yi kur

In [ ]:
vectordb = Chroma.from_documents(chunks, embeddings = embeddings, collection_name = "demo")
print(vectordb_collection.count(), "chunk vektör DB'ye eklendi")

# 3.1 Vektör DB'de ne var? (içine bakalım)

- Kaydettiğimiz her şeyi görelim: chunk'ın metni, kategorisi ve dönüştüğü vektör. Soyut "vektör DB" böyle somutlaşır -- get() ile çekiyoruz.

In [ ]:
import pandas as pd
data = vectordb.get(include = ["documents","metadatas","embeddings"])
df = pd.read_csv({
    "id": [i[:8] for i in data["ids"]],
    "kategori": [m["kategori"] for m in data["metadatas"]],
    "içerik (ilk 50)" : [df[50].replace("\n"," ") for d in data["documents"]],
    "vektör_boyut": [len(e) for e in data["embeddings"]],
    "vektör[:3]": [[round(float(x),2) for x in e[:3]] for e in data["embeddings"]]
})
print("Toplam chunk:", vectordb._collection.count())
df.head()

# 4. Anlamsal Arama (kelime değil, ANLAM)

- Dikkat: sorguda "iade" kelimesi geçmiyor ama embdedding anlamı yakaladığı için doğru belgeyi buluyor.

In [ ]:
for r in vectordb.similarity_search("ürünün geri gönderebilir miyim?", k = 2):
    print(f"[{r.metadata["kategori"]}]. {r.page_content[:70].strip()}...")

In [ ]:
for doc, skor in vectordb.similarity_search_with_score("kargo ne zaman bedava?", k = 2):
    print(f"Skor={skor:.3f} [{doc.metadata["kategori"]}]. {doc.page_content[:55].strip()}...")

# 5. Metadata ile filtreleme

- Sadece Kargo & Teslimat belgesinden ara gibi -- semantik arama + yapısal filtre birlikte

In [ ]:
for r in vectordb.similarity_search("teslimat süresi", k = 2, filter = {"kategori": "Kargo & Teslimat"}):
    print(f"{r.metadata['kategori']} {r.page_content[:60].strip()}...")

# Özet

- Vektör DB -> fast similarity search at scale

- Chroma: kolay başlangıç, local dev

- Dökümanlar + metadata -> filtered search

- persist_directory ile disk backup

- Production: Pinecone / Qrant / PGVector

# ChromeDB Kurulumu


In [ ]:
# pip install langchain-chroma chromadb

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

# In memory 
store = Chroma(
    colleciton_name = "sepetix",
    embeddings_function = embeddings
)

# Dökümanlar Ekleme

In [ ]:
from langchain_core.document import Document

docs = [
    Document("Ürünü 14 gün içinde iade edebilirsiniz", metadata = {"kategori": "Kargo & Teslimat"}),
    Document("250 TL üzeri siparişlerde kargo ücretisizdir", metadata = {"kategori": "Kargo & Teslimat"}),
    Document("Üye olarak her alışverişte puan kazanırsınız", metadata = {"kategori": "Üyelik & Sadakat"})
]

store = Chroma.from_document(docs, embeddings, collection_names = "sepetix")

store.add_documents(new_docs)

# Arama 

In [ ]:
# Basit similarity
results = store.similarity_search("iade nasıl yapılır?", k = 3)

# Score İle
results = store.similarity_search_with_score("kargo ne zaman ücretsiz?", k = 3)
for doc, score in results:
    print(f"{score:3f}: {doc.page_content}")

# Metadata Filtering

In [ ]:
results = store.similarity_search(
    "ürünü geri gönderebilir miyim",
    k = 3,
    filter = {"kategori": "İade & Değişim"}
)

# Karmaşık filter
results = store.similarity_search(
    "teslimat",
    k = 3,
    filter = {
        "kategori": {"$in": ["Kargo & Teslimat", "İade & Değişim"]}
    }
)

# Persistent Storage

In [ ]:
# Diske Kaydet
store = Chroma.from_documents(
    docs, embeddings,
    persist_directory = "./chroma_db",
    collection_name = "sepetix"
)

# Sonraki çalıştormada tekrar yukle
store =Chroma(
    persist_directory = "./chroma_db",
    collection_name = "sepetix",
    embedding_functiob = embeddings
)